In [41]:
import os
from langgraph.graph import StateGraph, MessagesState, START, END
from typing import TypedDict, Annotated, List, Literal
import operator
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from IPython.display import Image, display

In [42]:
load_dotenv()

True

In [43]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [53]:
class SentimentState(TypedDict):
    text: str
    score: int

In [54]:
def sentiment_score(state: SentimentState):
    text = state["text"]
    response = llm.invoke(
        f"Analyze the sentiment of the following text and provide only a score from 1 to 10 and dont give anything else, where 1 is very negative and 10 is very positive: {text}"
    )
    score = int(response.content.strip())
    return {"score": score}

In [55]:
def conditional_node(
    state: SentimentState,
):
    score = state["score"]

    if score >= 7:
        return "positive_node"
    elif score <= 4:
        return "negative_node"
    else:
        return "neutral_node"

In [56]:
def positive_node(state: SentimentState):
    return {"final_output": f"The text is positive: {state['text']}"}

In [57]:
def negative_node(state: SentimentState):
    return {"final_output": f"The text is negative: {state['text']}"}

In [58]:
def neutral_node(state: SentimentState):
    return {"final_output": f"The text is neutral: {state['text']}"}

In [ ]:
workflow = StateGraph(SentimentState)
workflow.add_node("sentiment_score", sentiment_score)
workflow.add_node("positive_node", positive_node)
workflow.add_node("negative_node", negative_node)
workflow.add_node("neutral_node", neutral_node)

workflow.add_edge(START, "sentiment_score")

workflow.add_conditional_edges(
    "sentiment_score",
    conditional_node,
    {
        "positive_node": "positive_node",
        "negative_node": "negative_node",
        "neutral_node": "neutral_node",
    },
)
workflow.add_edge("positive_node", END)
workflow.add_edge("negative_node", END)
workflow.add_edge("neutral_node", END)
app = workflow.compile()

In [61]:
app.invoke({"text": "I love this product! It's amazing."})

{'text': "I love this product! It's amazing.", 'score': 10}